# RSNA Knee Abnormality Detection — v2 (adaptive MedSLipMedSigLIP + Gemma report cache)

Fully offline; top-to-bottom like the v1 kernel; the 0.5 benchmark submission
is written before any model runs.

1. **vision**: adaptive encoder behind one `encode()` interface — HF
   `SiglipVisionModel` dir, KerasHub preset dir (config.json + weights.h5),
   or a learned stub. Each branch is smoke-tested with a zero-image forward;
   the first that passes is kept.
2. **text**: Gemma report embeddings (mean-pooled last hidden state,
   L2-normalised) computed once per split and cached next to the vision
   cache; broadcast per slot and concatenated to the vision features.
3. **head**: v1's SlotHead untouched; EMA is a shadow of trainable weights
   (never deepcopies a possibly-Keras backbone); the averaged snapshot is
   loaded into a fresh copy for validation and submission.

In [ ]:
import os, sys, re, json, glob, gc, time, hashlib, warnings, subprocess

import traceback, math, unicodedata

from copy import deepcopy

from pathlib import Path

from concurrent.futures import ThreadPoolExecutor



import numpy as np

import pandas as pd

import pydicom

import torch

import torch.nn as nn

import torch.nn.functional as F



warnings.filterwarnings("ignore")

np.random.seed(2026)

torch.manual_seed(2026)



T0 = time.time()



def log(msg):

    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


## Step 1 — labels (inline multilingual rule extractor, fully offline)

In [ ]:
# ---- TARGETS -----------------------------------------------------------

TARGETS = [

    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",

    "Medial OA", "Lateral OA", "PF OA", "Effusion",

    "Synovitis", "Baker's", "Contusion", "Fracture",

]


In [ ]:




# Turkish dotted/dotless i must be folded before casefolding, otherwise "İZLENMEZ"

# and "izlenmez" diverge. ß and the Croatian/Serbian d-with-stroke likewise.

_PRE = str.maketrans({

    "ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d",

    "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae",

})





def normalize(text: str) -> str:

    """Fold case, diacritics and separators; keep Greek and Cyrillic letters.



    NFKD decomposition strips Latin accents and Greek tonos alike (ά -> α), which is what

    we want: reports are inconsistent about accents. It also maps the MICRO SIGN U+00B5

    to a real mu, which matters because most Greek reports here use the wrong codepoint.

    """

    if not isinstance(text, str):

        return ""

    text = text.translate(_PRE).lower()

    text = unicodedata.normalize("NFKD", text)

    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    text = text.replace("­", "")                    # soft hyphen

    text = re.sub(r"[_\-/\\]+", " ", text)

    text = re.sub(r"[ \t]+", " ", text)

    return text





_SENT_SPLIT = re.compile(r"(?<=[.;!?])\s+|\n+")





def clauses(text: str):

    """Split into clauses, then attach `header:` lines to the value that follows.



    A report line reading `Fractures :` followed by `Aucune.` is one statement. Splitting

    on punctuation alone separates the anatomy from its negation and flips the label.

    """

    norm = normalize(text)

    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]



    merged = []

    for i, c in enumerate(raw):

        # A fragment ending in a colon is a heading for the next fragment. Structured

        # English reports write long ones - "lateral compartment (meniscus, collateral

        # ligament complex, cartilage):" is eight words - so the cap is generous.

        #

        # A merged heading must NOT also stand alone. On its own it carries the anatomy

        # word with no negation in scope, so `Fractures :` / `Aucune.` asserted a fracture

        # off the heading while the joined clause correctly read the denial. The joined

        # clause is a superset of the heading, so nothing is lost by dropping it; a

        # heading with no value beneath it is not merged and still stands.

        if c.endswith(":") and len(c.split()) <= 14 and i + 1 < len(raw):

            merged.append(c + " " + raw[i + 1])

        else:

            merged.append(c)

    # Comma-separated enumerations inside a long clause hide separate assertions.

    out = []

    for c in merged:

        out.append(c)

        if len(c.split()) > 25:

            out.extend(p.strip() for p in c.split(",") if len(p.split()) > 2)

    return out





def _rx(*alts: str) -> re.Pattern:

    return re.compile("|".join(alts))





NEGATION = _rx(

    # en

    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative for\b", r"\babsence\b",

    r"\bno evidence\b", r"\bunremarkable\b", r"\bfree of\b", r"\bnone\b", r"\bnil\b",

    # es

    r"\bsin\b", r"\bno hay\b", r"\bausencia\b", r"\bausentes?\b",

    # fr

    r"\bpas de\b", r"\bsans\b", r"\baucune?\b", r"\babsence\b",

    # nl

    r"\bgeen\b", r"\bzonder\b", r"\bniet\b",

    # de

    r"\bkeine?\b", r"\bohne\b", r"\bnicht\b",

    # tr

    r"\byok\b", r"\byoktur\b", r"izlenmemekte", r"saptanmadi", r"\bdegil\b",

    r"gozlenmemekte", r"mevcut degil", r"eslik etmiyor", r"\bizlenmedi\b",

    # hr / sr / bs

    r"\bnema\b", r"\bbez\b", r"\bnisu\b", r"\bnije\b",

    # el (accents already stripped)

    r"\bδεν\b", r"\bχωρις\b", r"ουδεν",

    # bg / ru

    r"\bбез\b", r"\bне\b", r"липсва", r"\bняма\b",

)



NORMALITY = _rx(

    r"\bnormal", r"\bintact\b", r"\bpreserved\b", r"\bwithin normal limits\b",

    r"limites normales", r"\bconservad", r"\bintegr", r"\bnormales\b",

    r"\bdoga(l|ll)\b", r"korunmus", r"\bnormaldir\b", r"olagan",

    r"\buredn", r"\bocuvan", r"\bodrzan", r"\bintakt",

    r"φυσιολογικ", r"ακεραι",

    r"unauffallig", r"regelrecht", r"\bintakt\b",

    r"нормал", r"запазен", r"съхранен", r"\bбез особености\b",

    r"\bgaaf\b", r"\bnormaal\b",

)



UNCERTAIN = _rx(

    r"\bpossible\b", r"\bprobable\b", r"\bsuspicious\b", r"\bsuspected\b",

    r"cannot (be )?exclude", r"\bmay\b", r"\bquestionable\b", r"\bequivocal\b",

    r"\bposible\b", r"sin criterios categoricos", r"\bdudos",

    r"\bmuhtemel\b", r"\bolasi\b", r"\bsupheli\b", r"\bizlenim",

    r"\bmoguce\b", r"\bvjerojatno\b", r"\bsumnja\b",

    r"πιθαν", r"υποπτ",

    r"\bmoglich", r"\bverdachtig", r"\bfraglich", r"\bV\.a\.\b",

    r"\bвъзможно\b", r"\bвероятно\b", r"суспект",

    r"\bmogelijk\b", r"\bverdacht\b",

)



# Pathology vocabulary shared by the paired rules.

TEAR = _rx(

    r"\btear", r"\btorn\b", r"\brupture", r"\bdisruption\b", r"discontinuit",

    r"\bavuls",

    r"\brotura\b", r"\broturas\b", r"\bruptura", r"\bdesgarro", r"\broto\b",

    r"\bdechirure", r"\bdechire",

    r"\bscheur", r"\bruptuur", r"gescheurd",

    r"\briss\b", r"einriss", r"\bruptur", r"zerreiss", r"\blasion",

    r"\byirtik", r"\byirtig", r"\bkopma\b", r"butunluk kaybi", r"\brupturu\b",

    r"\bpuknuce", r"\bruptur", r"\bprekid\b", r"\bpukotin",

    r"ρηξη", r"ρηξις", r"ρηγμα",

    r"руптура", r"разкъсв", r"разрив", r"скъсв",

)



DEGEN = _rx(

    r"degenerat", r"\bmucoid\b", r"\bmyxoid\b", r"\bfray", r"\bfissur",

    r"dejeneratif", r"\bmukoid\b", r"degenerativn", r"εκφυλ", r"дегенерат",

    r"\bμυξοειδ", r"\bμυξωδ",

    r"\bmuco ?ide\b", r"aufgefasert",

)



INJURY = _rx(

    r"\binjur", r"\bsprain", r"\blesion", r"\blasion", r"\bedema\b", r"\boedema\b",

    r"\bodem\b", r"\bedem\b", r"\bοιδημα", r"\bодем", r"\bедем", r"\bstrain\b",

    r"\bhigh signal\b", r"\bsignal alteration\b", r"\bhiperintens", r"\bhyperintens",

    r"aumento de senal", r"alteracion de senal", r"cambio de senal",

    r"\bsignalanhebung", r"\bsignalalteration", r"verhoogd signaal", r"sinyal artis",

    r"αυξημενο σημα", r"повишен сигнал",

    r"\bthicken", r"\bzadebljanje\b", r"\bverdikking\b", r"\bdistenzij",

    r"\blaksite\b", r"\blaxity\b", r"\bpartial\b", r"\bparcijaln", r"\bparcial",

    r"\bpartiel", r"\bpartiell",

)





ANAT = {

    "ACL": _rx(

        r"anterior cruciate", r"\bacl\b",

        r"cruzado anterior", r"\blca\b",

        r"croise anterieur",

        r"voorste kruisband", r"\bvkb\b",

        r"vorderes kreuzband", r"vorderen kreuzband", r"vordere kreuzband",

        r"on capraz", r"\bocb\b",

        r"prednji krizni", r"prednjeg krizn",

        r"προσθι[οα][^ ]* χιαστ", r"προσθιου χιαστου", r"χιαστο[^ ]* συνδεσμ",

        # "χιαστοι και πλαγιοι συνδεσμοι" separates the adjective from its noun, so the

        # adjective stem has to stand alone. Greek marks cruciate with it unambiguously.

        r"\bχιαστ\w*",

        r"предна кръстна", r"предната кръстна",

        # Plural, unqualified: reports routinely clear both cruciates in one clause

        # ("Ligamentos cruzados y colaterales dentro de limites normales"), so the

        # plural form has to match without a side qualifier or the whole clause is lost.

        r"cruciate ligaments", r"ligamentos cruzados", r"ligaments croises",

        r"kruisbanden", r"kreuzbander", r"capraz baglar", r"krizn[a-z]* ligament[a-z]*",

        r"χιαστοι συνδεσμ", r"χιαστων συνδεσμ", r"кръстните връзки", r"кръстни връзки",

    ),

    "MCL": _rx(

        r"medial collateral", r"\bmcl\b", r"tibial collateral",

        r"colateral medial", r"colateral interno", r"\blcm\b",

        r"collateral medial", r"collateral interne",

        r"mediale collaterale", r"binnenband", r"\b(mediale|laterale) banden\b",

        r"\bcollaterale banden\b",

        r"innenband", r"mediales? kollateral",

        r"\bic yan bag", r"medial kollateral", r"\biyb\b",

        r"medijalni kolateraln", r"medijalnog kolateraln",

        r"εσω πλαγι", r"εσωτερικο πλαγι", r"\bπλαγι\w* συνδεσμ", r"\bπλαγιοι\b",

        r"медиален колатерал", r"вътрешна странична", r"\bколатерал\w*",

        # Same plural pattern as the cruciates.

        # "Ligamentos cruzados y colaterales" separates the noun from its adjective, so

        # the adjective has to stand alone as a cue.

        r"\bcolaterales\b", r"\bcollateraux\b", r"\bcollateralen\b", r"\bkolateralni\b",

        r"collateral ligaments", r"ligamentos colaterales", r"ligaments collateraux",

        r"collaterale banden", r"kollateralbander", r"seitenbander", r"yan baglar",

        r"kolateraln[a-z]* ligament[a-z]*", r"πλαγιοι συνδεσμ", r"πλαγιων συνδεσμ",

        r"колатерални връзки", r"страничните връзки",

    ),

    "Medial Meniscus": _rx(

        r"medial meniscus", r"\bmm\b(?= tear)", r"medial menisc",

        r"menisco medial", r"menisco interno",

        r"menisque medial", r"menisque interne",

        r"mediale meniscus", r"binnenmeniscus",

        r"innenmeniskus", r"medialen? meniskus", r"innenmeniskushinterhorn",

        r"medyal menisk", r"\bic menisk",

        r"medijalni meniskus", r"medijalnog meniskusa", r"medijalnom meniskusu",

        r"εσω μηνισκ", r"μηνισκ[^ ]* του εσω", r"εσω διαμερισμα[^.]{0,40}μηνισκ",

        r"медиалния менискус", r"медиален менискус", r"вътрешния менискус",

    ),

    "Lateral Meniscus": _rx(

        r"lateral meniscus", r"lateral menisc",

        r"menisco lateral", r"menisco externo",

        r"menisque lateral", r"menisque externe",

        r"laterale meniscus", r"buitenmeniscus",

        r"aussenmeniskus", r"lateralen? meniskus",

        r"lateral menisk", r"\bdis menisk",

        r"lateralni meniskus", r"lateralnog meniskusa", r"lateralnom meniskusu",

        r"εξω μηνισκ", r"μηνισκ[^ ]* του εξω", r"εξω διαμερισμα[^.]{0,40}μηνισκ",

        r"латералния менискус", r"латерален менискус", r"външния менискус",

    ),

}



# Osteoarthritis is rarely written as "osteoarthritis". It is written as cartilage loss,

# chondropathy grade, joint space narrowing, or osteophytes - scoped to a compartment.

OA_EVIDENCE = _rx(

    r"osteoarthrit", r"\barthros", r"\bgonarthros", r"\bosteoarthros",

    r"chondropath", r"chondromalac", r"condropat", r"condromalac",

    r"cartilage loss", r"cartilage thinning", r"chondral (loss|defect|ulcer|thinning)",

    r"osteophyt", r"osteofit", r"osteofyt", r"osteofito", r"osteophyten",

    r"joint space narrowing", r"pinzamiento articular",

    r"kikirdak kayb", r"kikirdak incelme", r"kondropati", r"kondral",

    r"kraakbeen(lijden|verlies)", r"gonartrose", r"artrose",

    r"knorpel(verlust|schaden|defekt)", r"arthrose", r"gonarthrose",

    r"hrskavic", r"hondromalac", r"artroz", r"osteoartrit",

    r"χονδρ[^ ]*παθ", r"αρθριτ", r"αρθρωσ", r"οστεοφυτ",

    r"αρθρικου χονδρου", r"εξαλειψη του αρθρικου χονδρου",

    r"артроз", r"хондропат", r"остеофит", r"хрущял[^.]{0,30}(изтън|увред|дефект)",

    r"ulcera[s]? condral", r"cartilago[^.]{0,25}(perdida|adelgaz)",

    r"icrs grade", r"outerbridge",

)



COMPARTMENT = {

    "Medial OA": _rx(

        r"medial (femorotibial|tibiofemoral|compartment)",

        r"compartimento femorotibial medial", r"femorotibial interno",

        r"mediaal femorotibiaal", r"mediale femorotibial",

        r"medial femorotibial", r"medialen kompartiment", r"innere[sn]? kompartiment",

        r"medyal femorotibial", r"ic kompartman", r"medyal kompartman",

        r"medijaln[^ ]* (femorotibi|odjelj|kompartm)",

        r"εσω διαμερισμα", r"εσω κνημιαι", r"εσω μηριαι",

        r"медиалн[^ ]* (компартм|отдел|тибиал|феморотиб)",

        r"medial (femoral|tibial) (condyle|plateau)", r"condilo femoral medial",

        r"medialen? (femurkondyl|tibiaplateau)", r"mediale femorale condyl",

    ),

    "Lateral OA": _rx(

        r"lateral (femorotibial|tibiofemoral|compartment)",

        r"compartimento femorotibial lateral", r"femorotibial externo",

        r"lateraal femorotibiaal", r"laterale femorotibial",

        r"lateral femorotibial", r"lateralen kompartiment", r"aussere[sn]? kompartiment",

        r"lateral femorotibial", r"dis kompartman", r"lateral kompartman",

        r"lateraln[^ ]* (femorotibi|odjelj|kompartm)",

        r"εξω διαμερισμα", r"εξω κνημιαι", r"εξω μηριαι",

        r"латералн[^ ]* (компартм|отдел|тибиал|феморотиб)",

        r"lateral (femoral|tibial) (condyle|plateau)", r"condilo femoral lateral",

        r"lateralen? (femurkondyl|tibiaplateau)", r"laterale femorale condyl",

    ),

    "PF OA": _rx(

        r"patellofemoral", r"femoropatellar", r"femoropatelar", r"patelofemoral",

        r"retropatellar", r"retrorotulian", r"\btrochlea", r"\btroclea", r"\btroklea",

        r"\bpatella\b", r"\bpatellar\b", r"\brotulian", r"\brotula\b", r"\bpatele\b",

        r"\bpatellae?\b", r"patellofemoraal", r"femoropatellair",

        r"επιγονατιδ", r"μηροεπιγονατιδ", r"τροχιλ",

        r"пател", r"феморопател", r"тролх",

        r"anterior compartment", r"compartimento anterior", r"prednj[^ ]* odjeljk",

    ),

}



# Self-declaring findings: the term itself is the finding.

DIRECT = {

    "Effusion": _rx(

        r"\beffusion", r"joint fluid", r"intra ?articular fluid", r"\bhydrops\b",

        r"derrame articular", r"\bderrame\b", r"liquido articular",

        r"epanchement",

        r"gewrichtsvocht", r"\bvocht\b", r"\bhydrops\b", r"gewrichtseffusie",

        r"gelenkerguss", r"\berguss\b", r"gelenksergu",

        # "diz eklemi ici sivi miktari ... artmis" and "eklem icerisinde yaygin sivi

        # artisi" both occur; the noun takes a possessive suffix, so `eklem ` alone

        # misses. Match the stem plus any suffix.

        r"eklem\w* ic\w* sivi", r"efuzyon", r"eklem sivisi",

        r"sivi (miktari|artisi|birikimi)", r"sivi artis", r"\bsivi\b[^.]{0,25}artmis",

        r"\bizljev", r"\bizliv", r"zglobn[^ ]* tekucin", r"\bhidrops\b",

        r"αρθρικ[^ ]* υγρ", r"υγρου ενδαρθρικα", r"ενδαρθρικ[^ ]* υγρ", r"ποσοτητα υγρου",

        r"ενδαρθρικ", r"αρθρικη συλλογη", r"υγρο στην αρθρωση", r"υγρου στην αρθρωση",

        r"ставен излив", r"излив", r"ставна течност", r"синовиална течност",

    ),

    "Synovitis": _rx(

        r"synovit", r"sinovit", r"synovial (thickening|proliferation|hypertroph)",

        r"synovitis", r"synoviale? (verdikking|proliferatie)",

        r"synovialitis", r"synovialis(verdickung|proliferation)",

        r"sinovijalitis", r"sinovitis", r"zadebljanje sinovij",

        r"υμενιτιδα", r"συνοβιτιδα", r"υμενικ[^ ]* υπερτροφ", r"αρθρικου υμεν",

        r"синовит", r"синовиал[^ ]* (задебел|пролифер)",

        r"verdikkingen van (het )?synovium", r"pannus",

    ),

    "Baker's": _rx(

        r"baker", r"popliteal cyst", r"quiste popliteo", r"quistes popliteos",

        r"kyste poplite", r"popliteale? cyst", r"poplitealzyste", r"bakerzyste",

        r"popliteal kist", r"\bbakerova\b", r"poplitealn[^ ]* cist",

        r"κυστη baker", r"πολυχωρη συνοβιακη κυστη", r"κυστη του baker",

        r"киста на бейкър", r"бейкърова киста", r"поплитеална киста",

        r"gastrocnemio ?semimembranos", r"gastrocnemius semimembranosus burs",

    ),

    "Contusion": _rx(

        r"\bcontusion", r"bone bruise", r"bone marrow (o?edema|contusion)",

        r"\bkontuz", r"medular bone o?edema", r"marrow o?edema",

        r"contusion osea", r"edema oseo", r"edema de medula osea",

        r"oedeme osseux", r"contusion osseuse",

        r"botcontusie", r"botoedeem", r"beenmergoedeem", r"botmergoedeem",

        r"knochenmarkodem", r"knochenodem", r"kontusion", r"bone bruise",

        r"kemik kontuzyonu", r"kemik iligi odemi", r"kemik odemi",

        r"kostani edem", r"edem kosti", r"kontuzij",

        r"οστεομυελικ[^ ]* οιδημα", r"οστικο οιδημα", r"μυελικο οιδημα",

        r"костномозъчен едем", r"костен едем", r"контузионен",

    ),

    "Fracture": _rx(

        r"\bfractur", r"\bfract\b",

        r"\bfractura", r"\bfracturas\b",

        r"\bfractuur", r"\bbreuk\b",

        r"\bfraktur", r"\bbruch\b",

        r"\bkirik\b", r"\bkirigi\b", r"\bkirik\b",

        r"\bfraktur", r"\bprijelom", r"impresijsk[^ ]* fraktur",

        r"καταγμα", r"καταγματ",

        r"фрактур", r"счупван", r"фисур",

        r"insufficiency fracture", r"stress fracture", r"avulsion fracture",

        r"subchondral fracture", r"subkondral kiri",

    ),

}



# Terms that look like a finding but are not the finding being scored.

DECOY = {

    # `no fracture` is deliberately absent: a decoy skips the clause, so listing it here

    # turned the commonest English denial into silence, and the study then pulled on the

    # fracture head with the weight of a report that never mentioned fractures at all.

    # `microfractur` is a surgical procedure and `fracture risk` a prediction; both stay.

    "Fracture": _rx(r"microfractur", r"\bfracture (risk|prophyla)"),

    "Baker's": _rx(r"meniscal cyst", r"quiste meniscal", r"ganglion"),

}



PAIRED = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}

OA_TARGETS = {"Medial OA", "Lateral OA", "PF OA"}





STEM_MENISCUS = _rx(r"menisc\w*", r"menisk\w*", r"μηνισκ\w*", r"мениск\w*")

STEM_CRUCIATE = _rx(r"cruciate", r"cruzado", r"croise", r"kruisband", r"kreuzband",

                    r"capraz bag\w*", r"krizn\w*", r"χιαστ\w*", r"кръстн\w*",

                    r"\bacl\b", r"\bpcl\b", r"\blca\b", r"\blcp\b", r"\bvkb\b",

                    r"\bhkb\b", r"\bocb\b", r"\bacb\b")

STEM_COLLATERAL = _rx(r"collateral\w*", r"colateral\w*", r"kollateral\w*",

                      r"collaterale\w*", r"kolateraln\w*", r"yan bag\w*",

                      r"πλαγι\w*", r"колатерал\w*", r"странич\w*",

                      r"innenband\w*", r"aussenband\w*", r"binnenband\w*",

                      r"\bmcl\b", r"\blcl\b", r"\blcm\b", r"\biyb\b")



SIDE_MEDIAL = _rx(r"\bmedial\w*", r"\bmedyal\w*", r"\bmedijaln\w*", r"\bmediaal\w*",

                  r"\bmediale\w*", r"\bintern[oa]\w*", r"\binterne\w*", r"\binnen\w*",

                  r"\bic\b", r"\bunutarnj\w*", r"\bεσω\w*", r"\bεσωτερικ\w*",

                  r"\bмедиал\w*", r"\bвътреш\w*", r"\btibial collateral\b",

                  r"\bbinnen\w*", r"\bmediaal\b")

SIDE_LATERAL = _rx(r"\blateral\w*", r"\bextern[oa]\w*", r"\bexterne\w*", r"\bdis\b",

                   r"\blateraln\w*", r"\baussen\w*", r"\bbuiten\w*", r"\bεξω\w*",

                   r"\bεξωτερικ\w*", r"\bлатерал\w*", r"\bвъншн\w*",

                   r"\bfibular collateral\b", r"\bvanjsk\w*")

SIDE_ANTERIOR = _rx(r"\banterior\w*", r"\bant\b", r"\bon\b", r"\bprednj\w*",

                    r"\bvorder\w*", r"\bvoorste\b", r"\bπροσθι\w*", r"\bпредн\w*",

                    r"\banteriyor\w*", r"\bavant\b", r"\bant[eé]rieur\w*")



# The contrary of SIDE_ANTERIOR, needed only to stop a side-blind cruciate cue firing on

# the posterior ligament. It is never used to assert a target - there is no PCL target -

# so it is deliberately narrow: `posterior horn` is one of the commonest phrases in a

# knee report and must not be read as a cruciate qualifier, which is why the guard below

# tests proximity to the cruciate stem rather than presence in the clause.

SIDE_POSTERIOR = _rx(r"\bposterior\w*", r"\bpost[eé]rieur\w*", r"\bposteriore\w*",

                     r"\bhinter\w*", r"\bachterste\b", r"\barka\b", r"\bstraznj\w*",

                     r"\bzadnj\w*", r"\bοπισθι\w*", r"\bзадн\w*", r"\bpostero\w*")



# Fracture is the target whose stem varies most across the corpus.

STEM_FRACTURE = _rx(r"fractur\w*", r"fraktur\w*", r"fractuur\w*", r"\bfract\b",

                    r"kiri[kgğ]\w*", r"prijelom\w*", r"lom kosti", r"\bbreuk\w*",

                    r"\bbruch\w*", r"καταγμα\w*", r"καταγματ\w*", r"фрактур\w*",

                    # NOT a bare `fissur\w*`: "fisuras condrales" and "full thickness

                    # fissures in the articular cartilage" describe cartilage, not bone.

                    # The stem has to be anchored to a bone word to mean a fracture.

                    r"счупван\w*", r"fisur\w* (osea|oseas|kost)", r"fissur\w* kost")



STEM_OA_COMPARTMENT = _rx(r"compartment\w*", r"compartimento\w*", r"compartiment\w*",

                          r"kompartman\w*", r"kompartiment\w*", r"odjelj\w*",

                          r"διαμερισμα\w*", r"компартм\w*", r"\bотдел\w*",

                          r"femorotibial\w*", r"femorotibiaal\w*", r"tibiofemoral\w*",

                          r"femoro tibial\w*", r"κνημιαι\w*", r"μηριαι\w*",

                          r"femoral condyl\w*", r"tibial plateau\w*",

                          r"condilo femoral", r"platillo tibial", r"tibiaplateau\w*",

                          r"femurkondyl\w*", r"femoralne? kondil\w*",

                          r"tibijaln\w* plato", r"femoral kondil\w*",

                          r"tibia plato", r"tibyal plato")





def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):

    """True if a stem match has a qualifier within `window` characters either side.



    Character windows rather than token windows, because word order differs: English

    puts the side before the noun, Greek and Bulgarian often after, and Turkish

    attaches it as a separate preceding adjective.

    """

    for m in stem_rx.finditer(clause):

        lo = max(0, m.start() - window)

        hi = min(len(clause), m.end() + window)

        if qual_rx.search(clause[lo:hi]):

            return True

    return False





# concept -> (stem, side) pairs used in addition to the phrase lexicons above

STEM_RULES = {

    "ACL": (STEM_CRUCIATE, SIDE_ANTERIOR),

    "MCL": (STEM_COLLATERAL, SIDE_MEDIAL),

    "Medial Meniscus": (STEM_MENISCUS, SIDE_MEDIAL),

    "Lateral Meniscus": (STEM_MENISCUS, SIDE_LATERAL),

    "Medial OA": (STEM_OA_COMPARTMENT, SIDE_MEDIAL),

    "Lateral OA": (STEM_OA_COMPARTMENT, SIDE_LATERAL),

}





SEV_LOW = _rx(

    r"\bsmall\b", r"\bminimal\b", r"\btrace\b", r"\bmild\b", r"\bslight\b",

    r"\btiny\b", r"\bscant\b", r"\bmimimal\b", r"\bdiscrete\b", r"\bfocal\b",

    r"\bleve\b", r"\bminim", r"\bpeque", r"\bligero\b", r"\bescaso\b", r"\bdiscreto\b",

    r"\bhafif\b", r"\bminimal\b", r"\baz miktarda\b", r"\bsilik\b",

    r"\bmanja\b", r"\bmanji\b", r"\bblago\b", r"\bdiskretn", r"\bmalo\b",

    r"\bgering", r"\bdiskret", r"\bkleine?r?\b", r"\bwenig\b", r"\bzarte?\b",

    r"\bbeperkte?\b", r"\bgeringe\b", r"\bweinig\b", r"\blichte?\b",

    r"\bηπι", r"\bμικρ", r"\bελαχιστ",

    r"\bминимал", r"\bлек", r"\bмалк", r"\bнеголям",

)



SEV_HIGH = _rx(

    r"\blarge\b", r"\bmarked\b", r"\bmassive\b", r"\bsevere\b", r"\bextensive\b",

    r"\bmoderate\b", r"\bgross\b", r"\bsignificant\b", r"\babundant\b", r"\btense\b",

    r"\bmoderad", r"\bimportante\b", r"\bsevera?\b", r"\bmarcad", r"\bcuantios",

    r"\bbelirgin\b", r"\byaygin\b", r"\bileri\b", r"\bciddi\b", r"\bbol\b",

    r"\bopsezan\b", r"\bveliki\b", r"\bizrazit", r"\bznacajn", r"\bumjeren",

    r"\bausgepragt", r"\bdeutlich", r"\bmassiv", r"\bmassig", r"\bgross",

    r"\buitgebreid", r"\bgevorderd", r"\bveel\b", r"\bmatige?\b",

    r"\bμετρι", r"\bμεγαλ", r"\bεκτεταμεν", r"\bευμεγεθ", r"\bσοβαρ",

    r"\bголям", r"\bизразен", r"\bзначим", r"\bумерен", r"\bобилен",

)



# OA is often asserted for the whole joint rather than per compartment

# ("tricompartmental osteoarthritis", "gonarthrose", "incipient OA of all three

# compartments"). Those statements are evidence for all three OA targets.

GLOBAL_OA = _rx(

    r"tri ?compartment", r"all three compartment", r"global(ised)? (oa|osteoarthrit)",

    r"\bgonarthros", r"\bgonartros", r"\bgonarthrose", r"\bgonartrose",

    r"osteoarthritis of the knee", r"artrosis (de |)(la )?rodilla", r"knee osteoarthrit",

    r"\bdiz osteoartrit", r"\bgonartroz", r"artroza koljena",

    r"οστεοαρθριτιδα", r"αρθριτιδα του γονατος",

    r"артроза на колянната", r"гонартроз",

    r"degenerative joint disease", r"\bdjd\b",

)



# A bare "bone marrow oedema" is not a contusion when it sits under a cartilage

# defect: subchondral oedema beneath a worn compartment is reactive degenerative signal,

# and reading it as a bruise turns every osteoarthritic knee into a trauma case.

DEGENERATIVE_MARROW = _rx(

    r"subchondral", r"subcondral", r"subkondral", r"supkondraln", r"subchondraln",

    r"υποχονδρι", r"субхондрал", r"subchondrale?",

    r"\bcyst", r"\bquist", r"\bzyste\b", r"\bcistic", r"reactive", r"reactivo",

)



TRAUMA = _rx(

    r"\bbruise\b", r"\bcontusion", r"\bkontuz", r"\bcontusion osea\b",

    r"\btrauma", r"\bimpaction\b", r"\bpivot shift\b", r"\bkissing\b",

    r"\bacute\b", r"\bagudo\b", r"\bakut", r"\bpivot kaymasi\b",

    r"\bcontusion osseuse\b", r"\bbone bruise\b", r"\bbotcontusie\b",

    r"\bконтузион", r"\bμωλωπ", r"\bkontuzij",

)





def _polarity(clause: str, anchor_end: int) -> str:

    """Classify one clause as positive, negative or uncertain for a matched term.



    Scope is the whole clause. Clause segmentation already keeps statements short, and

    a window in characters mis-scopes badly across languages with different word orders -

    Turkish puts its negator at the end of the sentence, English at the front.

    """

    if UNCERTAIN.search(clause):

        return "uncertain"

    if NEGATION.search(clause):

        return "negative"

    if NORMALITY.search(clause):

        # "meniscus normal" negates; "normal ... but tear" does not.

        if TEAR.search(clause) or re.search(r"\bgrade [34]\b", clause):

            return "positive"

        return "negative"

    return "positive"





class _Matcher:

    """Phrase lexicon first, stem+side proximity as the fallback.



    Exposes `.search` so it drops into the same slot as a compiled pattern.

    """



    def __init__(self, phrase_rx, stem=None, side=None, window=55, contrary=None):

        self.phrase_rx = phrase_rx

        self.stem = stem

        self.side = side

        self.window = window

        self.contrary = contrary



    def search(self, clause):

        m = self.phrase_rx.search(clause)

        if m is not None and not self._wrong_side(clause):

            return m

        if self.stem is not None and _near(clause, self.stem, self.side, self.window):

            return self.stem.search(clause)

        return None



    def _wrong_side(self, clause):

        """True when the clause names the other member of this structure's pair.



        Some cues in the lexicon are side-blind by design: Greek separates the adjective

        from its noun ("cruciate and collateral ligaments"), so the bare adjective stem

        has to stand alone or the clause is lost. That stem then also matches the

        posterior cruciate and the lateral collateral, neither of which is a target here,

        and a positive outranks every negative in the scorer - so one PCL clause was

        enough to override an explicit "the ACL is normal".



        The test is proximity to the structure's own stem, not presence in the clause.

        "Posterior horn of the medial meniscus" appears in a large share of knee reports

        and says nothing about a cruciate; only a qualifier sitting beside the ligament

        word is one. A clause naming both sides keeps the match, because it does mention

        this target.

        """

        if self.contrary is None or self.stem is None:

            return False

        return (_near(clause, self.stem, self.contrary, self.window)

                and not _near(clause, self.stem, self.side, self.window))





# Which cue, if it sits beside the structure's stem, means the clause is about the other

# member of the pair. Only the two structures with a side-blind cue need one.

CONTRARY = {"ACL": SIDE_POSTERIOR, "MCL": SIDE_LATERAL}



ANAT_MATCH = {

    tgt: _Matcher(ANAT[tgt], *STEM_RULES[tgt], contrary=CONTRARY.get(tgt))

    for tgt in PAIRED

}

COMPARTMENT_MATCH = {

    "Medial OA": _Matcher(COMPARTMENT["Medial OA"], *STEM_RULES["Medial OA"]),

    "Lateral OA": _Matcher(COMPARTMENT["Lateral OA"], *STEM_RULES["Lateral OA"]),

    "PF OA": _Matcher(COMPARTMENT["PF OA"]),

}

DIRECT_MATCH = {

    tgt: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if tgt == "Fracture" else rx)

    for tgt, rx in DIRECT.items()

}





def _severity(clause: str) -> float:

    """Weight one positive mention by how emphatic the sentence is.



    Ordered, not calibrated. A "moderate effusion" must outrank a "trace effusion" and

    both must outrank silence; the absolute numbers do not matter to AUC.

    """

    high = SEV_HIGH.search(clause) is not None

    low = SEV_LOW.search(clause) is not None

    if high and not low:

        return 1.0

    if low and not high:

        return 0.45

    return 0.75                       # unqualified mention





def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None,

                   context_bonus=None):

    """Accumulate graded evidence over clauses for one target.



    Returns (score, confidence, n_pos, n_neg). Positives are graded by severity and by

    optional context regexes; negatives only matter when nothing positive was found,

    because reports assert normality for every structure they check.

    """

    n_pos = n_neg = n_unc = 0

    best = 0.0

    for c in cls:

        m = anat_rx.search(c)

        if not m:

            continue

        if decoy_rx is not None and decoy_rx.search(c):

            continue

        if path_rx is not None and not path_rx.search(c):

            if NORMALITY.search(c) and not NEGATION.search(c):

                n_neg += 1

            continue

        pol = _polarity(c, m.end())

        if pol == "positive":

            n_pos += 1

            w = _severity(c)

            if context_penalty is not None and context_penalty.search(c):

                w *= 0.45

            if context_bonus is not None and context_bonus.search(c):

                w = min(1.0, w * 1.35)

            best = max(best, w)

        elif pol == "negative":

            n_neg += 1

        else:

            n_unc += 1

            best = max(best, 0.30)



    if n_pos or n_unc:

        # 0.52 .. 0.95, ordered by the strongest single mention, nudged by repetition.

        score = min(0.95, 0.50 + 0.42 * best + 0.03 * min(n_pos, 3))

        conf = min(1.0, 0.55 + 0.15 * n_pos)

    elif n_neg:

        score = max(0.04, 0.20 - 0.04 * n_neg)

        conf = min(0.9, 0.45 + 0.12 * n_neg)

    else:

        score, conf = 0.28, 0.05          # silence sits above asserted-negative

    return score, conf, n_pos, n_neg





def extract(report: str) -> dict:

    """Extract twelve (score, confidence) pairs from one report."""

    cls = clauses(report)

    out = {}

    path_paired = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)



    for tgt in TARGETS:

        if tgt in PAIRED:

            s, c, npos, nneg = _score_clauses(cls, ANAT_MATCH[tgt], path_paired)

        elif tgt in OA_TARGETS:

            s, c, npos, nneg = _score_clauses(cls, COMPARTMENT_MATCH[tgt], OA_EVIDENCE)

        elif tgt == "Contusion":

            # Reactive subchondral oedema under a cartilage defect is osteoarthritis,

            # not a bruise. Explicit trauma wording pushes the other way.

            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt),

                                              context_penalty=DEGENERATIVE_MARROW,

                                              context_bonus=TRAUMA)

        else:

            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))

        out[tgt] = s

        out[tgt + "__conf"] = c

        out[tgt + "__npos"] = npos

        out[tgt + "__nneg"] = nneg



    # --- cross-target corrections ------------------------------------------ #

    # A whole-joint osteoarthritis statement is evidence for every compartment that was

    # not separately assessed. Without this, "incipient OA of all three compartments"

    # scores zero on all three OA targets.

    g_hits = [c for c in cls if GLOBAL_OA.search(c) and _polarity(c, 0) == "positive"]

    if g_hits:

        gscore = 0.50 + 0.42 * max(_severity(c) for c in g_hits)

        for tgt in OA_TARGETS:

            if out[tgt + "__npos"] == 0 and out[tgt + "__nneg"] == 0:

                out[tgt] = max(out[tgt], gscore * 0.92)

                out[tgt + "__conf"] = max(out[tgt + "__conf"], 0.4)



    # Synovitis is frequently visible on the images and absent from the text, so silence

    # is weak evidence of absence here in a way it is not for other findings. Effusion is

    # its most reliable textual proxy - the two share a mechanism - so a silent synovitis

    # inherits a fraction of the effusion evidence instead of falling to the floor.

    if out["Synovitis__npos"] == 0 and out["Synovitis__nneg"] == 0:

        out["Synovitis"] = max(out["Synovitis"], 0.28 + 0.45 * (out["Effusion"] - 0.28))



    return out


In [ ]:
# ---- configuration ------------------------------------------------

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):

    os.environ.setdefault(_v, "4")



CROP_MM = 130.0        # below the FOV of 99.6% of series; fixes pixel scale

IMG = 448              # MedSigLIP native resolution

GROUP = 3              # slices per encoder input, stacked as the RGB channels

CACHE_SLICES = 3       # only the first GROUP channels are used in v1

CACHE_FRACTION = 0.45

CACHE_BUDGET_MAX_GB = 24.0

HDR_THREADS = 16

PIX_THREADS = 12

ORDER_THREADS = 32

ORDER_BUDGET_S = 5400

TEST_SHARE = 0.30

SLICE_BAND = (0.20, 0.80)

LAT_MIN_OFFSET_MM = 20.0

LAT_MIN_AGREEMENT = 0.85

LAT_FALLBACK = "auto"



N_FOLDS = 4

EPOCHS = 8

BATCH_STUDIES = 8

AUG_ROT_DEG = 8.0

AUG_SCALE = 0.08

AUG_SHIFT = 0.05

AUG_INTENSITY = 0.10

USE_VFLIP = False

EMA_DECAY = 0.997

RANK_LOSS_W = 0.05

RANK_POS, RANK_NEG = 0.60, 0.40

GOLD_WEIGHT = 3.0

SELECTION = "worse_of_two"

LR_HEAD = 1e-3

WEIGHT_DECAY = 0.02

EVAL_BATCH = 8

TIME_BUDGET = 8.0 * 3600



SLOTS = [

    ("SAG_FLUID_FS", "Sagittal", True, True),

    ("COR_FLUID_FS", "Coronal", True, True),

    ("AX_FLUID_FS", "Axial", True, True),

    ("SAG_FLUID_NOFS", "Sagittal", True, False),

    ("COR_T1", "Coronal", False, False),

    ("SAG_T1", "Sagittal", False, False),

]

N_SLOT = len(SLOTS)



FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}

_SEP = re.compile(r"[_\-.]")

_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"

                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")

_T1_RX = re.compile(r"\bt1\b|\bt1w\b")

_T2_RX = re.compile(r"\bt2\b|\bt2w\b")

_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


In [ ]:
# ---- competition root (robust to nested mount) -------------------------

def find_root():

    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),

              Path("/kaggle/input/rsna-knee-abnormality-detection"),

              Path("data")]:

        if (c / "test.csv").is_file() and (c / "test_series").is_dir():

            return c

    base = Path("/kaggle/input")

    if base.is_dir():

        for d1 in sorted(p for p in base.iterdir() if p.is_dir()):

            for cand in [d1] + sorted(p for p in d1.iterdir() if p.is_dir()):

                if (cand / "test.csv").is_file():

                    return cand

    raise FileNotFoundError("competition mount not found; need a dir holding "

                            "test.csv and test_series/")



ROOT = find_root()



def available_gb():

    try:

        import psutil

        return psutil.virtual_memory().available / 1024 ** 3

    except Exception:

        return 16.0


In [ ]:
# ---- labels: inline extractor on every report ----------------------------

LABEL_COLS = TARGETS + [t + "__conf" for t in TARGETS]



def read_labels(train_df):

    lab = pd.DataFrame([extract(r) for r in train_df["Report"].fillna("")])

    lab["StudyInstanceUID"] = train_df["StudyInstanceUID"].values

    lab = lab.set_index("StudyInstanceUID")

    log(f"LABEL SOURCE: inline lexicon on {len(lab)} reports")

    return lab





def build_targets(st_tr, gold, lab):

    Y = np.zeros((len(st_tr), len(TARGETS)), np.float32)

    W = np.zeros_like(Y)

    for i, st in enumerate(st_tr):

        if st in gold.index:

            Y[i], W[i] = gold.loc[st].values, GOLD_WEIGHT

        elif st in lab.index:

            r = lab.loc[st]

            Y[i] = r[TARGETS].values

            W[i] = 0.25 + 0.75 * r[[t + "__conf" for t in TARGETS]].values

    return Y, W, np.where(W.sum(1) > 0)[0]


## Step 2 — DICOM → slots → cache

In [ ]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",

            "RepetitionTime", "EchoTime", "Laterality", "ImageLaterality",

            "ImagePositionPatient", "PixelSpacing", "Rows",

            "Columns", "RescaleSlope", "RescaleIntercept"]





def probe(item):

    split, study, series, path = item

    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,

           "dir": path}

    try:

        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))

        row["files"] = files

        row["n_slices"] = len(files)

        if not files:

            return row

        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),

                             stop_before_pixels=True, force=True)

        for t in HDR_TAGS:

            v = getattr(ds, t, None)

            if v is None:

                row[t] = None

            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":

                row[t] = "|".join(str(x) for x in v)

            else:

                row[t] = str(v)

    except Exception as exc:

        row["err"] = str(exc)[:120]

    return row





def walk(split):

    base = ROOT / split

    items = []

    if not base.is_dir():

        return pd.DataFrame()

    for study in os.scandir(base):

        if study.is_dir():

            for series in os.scandir(study.path):

                if series.is_dir():

                    items.append((split, study.name, series.name, series.path))

    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:

        rows = list(pool.map(probe, items))

    return pd.DataFrame(rows)





def annotate(df):

    """Recover fat suppression and pulse-sequence weighting from the header."""

    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))

    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)



    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")

    # GE writes SAT_GEMS for spatial saturation, so ScanOptions must be matched as

    # exact tokens; a substring test on "SAT" fires on non-fat-sat series.

    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))

    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs



    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")

    te = pd.to_numeric(df["EchoTime"], errors="coerce")

    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")

    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)



    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",

                     np.where(t2 & ~pdw, "T2",

                       np.where(pdw, "PD",

                         np.where(gre, "GRE",

                           np.where(tr < 800, "T1",

                             np.where(te > 60, "T2",

                               np.where(tr >= 800, "PD", "UNK")))))))

    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])

    df["px"] = pd.to_numeric(

        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),

        errors="coerce")

    return df





# --- laterality resolution -----------------------------------------------------------

# The tag is authoritative where it exists. Where it does not, the patient x-coordinate

# can stand in, but only if it agrees with the tag on the studies that have both: a wrong

# flip is worse than no flip, and knee coils often place the joint near isocentre, where

# the sign carries no information at all.



def _tag_side(g):

    v = [str(x).strip().upper() for x in g["Laterality"].dropna()]

    if "ImageLaterality" in g.columns:

        v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]

    v = [x[0] for x in v if x and x[0] in ("L", "R")]

    return v[0] if v else None





def _position_side(g):

    xs = []

    for s in g.get("ImagePositionPatient", pd.Series(dtype=object)).dropna():

        try:

            xs.append(float(str(s).split("|")[0]))

        except Exception:

            pass

    if not xs:

        return None

    x = float(np.median(xs))

    if abs(x) < LAT_MIN_OFFSET_MM:

        return None                      # centred in the coil: the sign means nothing

    return "R" if x < 0 else "L"         # LPS: the right knee sits at negative x





def laterality_maps(h):

    """Return (side_by_study, diagnostics). Uses the fallback only if it earns it."""

    tag, pos = {}, {}

    for st, g in h.groupby("StudyInstanceUID"):

        tag[st] = _tag_side(g)

        pos[st] = _position_side(g)



    both = [st for st in tag if tag[st] and pos[st]]

    agree = float(np.mean([tag[st] == pos[st] for st in both])) if both else np.nan

    have_tag = float(np.mean([v is not None for v in tag.values()]))



    if LAT_FALLBACK == "on":

        use = True

    elif LAT_FALLBACK == "off":

        use = False

    else:

        use = bool(both) and np.isfinite(agree) and agree >= LAT_MIN_AGREEMENT



    side = {st: (tag[st] or (pos[st] if use else None)) for st in tag}

    covered = float(np.mean([v is not None for v in side.values()]))

    info = {"tag_coverage": have_tag, "agreement": agree, "n_compared": len(both),

            "fallback_used": use, "final_coverage": covered}

    log(f"laterality: tag on {have_tag:.1%} of studies, x-sign agrees with it on "

        f"{agree:.1%} of {len(both)} comparable studies, fallback "

        f"{'enabled' if use else 'disabled'} -> {covered:.1%} normalised")

    return side, info

In [ ]:
def pick_slots(series_df, plane_map):

    """One series per slot per study.



    Ties are broken toward the stack with the most slices: a thicker stack samples the

    joint more densely, and the three-slice sampler below benefits from the margin.

    """

    series_df = series_df.copy()

    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)

    out = {}

    for study, g in series_df.groupby("StudyInstanceUID"):

        chosen = {}

        for name, plane, fluid, fs in SLOTS:

            sel = (g["plane"] == plane) & (g["fatsat"] == fs)

            # fluid=None means "do not condition on weighting" - the public scheme,

            # where the single provided flag stands in for both axes at once.

            if fluid is not None:

                sel &= (g["fluid"] == fluid)

            cand = g[sel]

            # A slot with no series matching its predicate stays empty, and no substitute

            # is admitted from a neighbouring predicate. Relaxing the weighting to fill a

            # T1 slot would draw from the pool `SAG_FLUID_NOFS` selects from, since that

            # pool is what remains once the weighting is dropped: over the training corpus

            # it would put one series in two slots for 2383 of 4407 studies and leave 56%

            # of the T1 slot holding PD or T2. The presence mask would then assert a

            # sequence that was never acquired, and the per-diagnosis softmax of §6 would

            # divide its attention across two identical slots, giving one acquisition

            # about twice the weight it carries in a study that holds both. The mask is

            # there to say a slot is absent, which is what an absent slot is.

            if len(cand):

                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]

        out[study] = chosen

    return out


In [ ]:
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]



# Series in which at least one sampled slice would not decode. A list rather than a

# counter because appending is atomic under the reader threads, and reported rather than

# swallowed: a decode failure used to be indistinguishable from a black knee.

DECODE_FAILED = []





def order_slices(rec):

    """Return the series' files sorted along the through-plane axis.



    A DICOM file name here is a SOP Instance UID, which is assigned arbitrarily. Sorting

    by it therefore produces an order uncorrelated with anatomy - measured over one

    series, Spearman between file-name rank and physical position is 0.009, i.e. none.

    Anything that assumes the file order means something is then operating on noise: the

    three channels of a "2.5D" input are three unrelated views rather than neighbouring

    slices, "the middle of the stack" is a random subset, and reversing slice order to

    normalise laterality reverses nothing meaningful.



    The physical order is recoverable exactly. Each slice carries its position in patient

    coordinates and the in-plane axes; projecting the position onto the slice normal

    gives a signed through-plane coordinate, monotonic along the stack:



        n = r_x  x  r_y ,      k = p . n



    `InstanceNumber` is the fallback. It usually tracks the projection up to sign, but

    interleaved and multi-echo acquisitions need not number slices in the order they

    occupy in space - but the projection is signed in patient

    coordinates, which is what laterality normalisation needs.

    """

    files, d = rec["files"], rec["dir"]

    keyed = []

    for f in files:

        k = None

        try:

            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,

                                 specific_tags=ORDER_TAGS)

            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)

            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)

            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))

        except Exception:

            try:

                k = float(ds.InstanceNumber)

            except Exception:

                k = None

        keyed.append((k, f))

    if any(k is None for k, _ in keyed):

        # A series with no usable geometry keeps its arbitrary order; that is worse than

        # sorting but better than dropping the series, and it is logged as a count.

        return files, False

    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True





def read_slot(rec, n_slice=None, out_size=None):

    """`n_slice` physically spread slices from one series, at `out_size` pixels.



    Returns uint8 [n_slice, out, out] normalised per-series to its 1st-99th

    percentile. Percentiles rather than min/max because MR intensity has no absolute

    scale and a single bright vessel would otherwise compress the whole dynamic range.



    Reading is the expensive half of this pipeline, so the caller reads once at the

    largest configuration it needs and derives the smaller ones from the returned buffer

    rather than re-reading.

    """

    n_slice = GROUP if n_slice is None else n_slice

    out_size = IMG if out_size is None else out_size

    files, d, px = rec.get("ordered") or rec["files"], rec["dir"], rec["px"]

    n = len(files)

    if n == 0:

        return None

    # Spread the samples over a central band of the stack: the outermost slices of a knee

    # series are mostly soft tissue outside the joint. The band is a constant rather than

    # a literal because how much of the stack is worth reading depends on how many slices

    # are being taken - at three the middle is all that fits, while at sixteen the ends

    # are worth having, and a Baker cyst sits at the posteromedial end of a sagittal one.

    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))

    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])

    while len(idx) < n_slice:

        idx = np.append(idx, idx[-1])



    planes = []

    for i in idx[:n_slice]:

        try:

            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)

            a = ds.pixel_array.astype(np.float32)

            sl = float(getattr(ds, "RescaleSlope", 1) or 1)

            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)

            a = a * sl + ic

        except Exception:

            a = None                      # no shape is known here; see below

        planes.append(a)



    # A slice that would not decode has no shape of its own, and inventing one is how a

    # single unreadable file could erase a whole series: the substitute used to be

    # allocated at the resize target while the slices that did decode were still native,

    # so the shape check below took the substitute as the authority and zeroed the good

    # slices with it. The result was a black slot that the presence mask still reported

    # as acquired.

    #

    # A failure is instead filled from the nearest slice that did decode - the same

    # convention the sampler already uses when the band holds fewer distinct slices than

    # were asked for - and a series where nothing decodes is reported absent, which the

    # mask can express, rather than black, which it cannot.

    got = [k for k, p in enumerate(planes) if p is not None]

    if not got:

        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))

        return None

    if len(got) < len(planes):

        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))

        for k, p in enumerate(planes):

            if p is None:

                planes[k] = planes[min(got, key=lambda j: abs(j - k))]



    # Slices of one series can still differ in matrix size - multi-echo and some

    # reformats do - and those are genuinely not stackable.

    shp = planes[0].shape

    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]

    vol = np.stack(planes)



    # constant physical extent, then resize: PixelSpacing varies 3.4x across the corpus

    if px and np.isfinite(px) and px > 0:

        want = int(round(CROP_MM / px))

        h, w = shp

        if 16 < want < min(h, w):

            cy, cx = h // 2, w // 2

            half = want // 2

            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]



    lo_v, hi_v = np.percentile(vol, [1, 99])

    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)



    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)

    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)

    # uint8, not float32. These buffers queue up between the reader threads and the

    # encoder, and at this size a float32 slot-series is several megabytes. Intensity is

    # already normalised into [0, 1] here, so eight bits cost nothing that a bilinear

    # resize has not already cost, and the queue is a quarter the size.

    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


In [ ]:
def normalise_laterality(img, plane, lat):

    """Map every knee onto a left-knee convention.



    Coronal and axial views mirror under a horizontal flip. Sagittal stacks are not

    mirror images of each other - the slice order runs medial-to-lateral in opposite

    directions - so the channel order is reversed instead.

    """

    if lat != "R":

        return img

    if plane in ("Coronal", "Axial"):

        return torch.flip(img, dims=[-1])

    return torch.flip(img, dims=[0])


In [ ]:
def build_cache(slot_map, plane_map, lat_map, tag):

    """Decode every (study, slot) once into an in-memory uint8 array.



    Fine-tuning revisits the same pixels every epoch. Reading them from the mount each

    time would make the epoch count a function of I/O rather than of learning, so they

    are decoded once and held as bytes: intensity has already been normalised into

    [0, 1], and eight bits cost nothing a bilinear resize has not already cost.



    CACHE_SLICES positions are kept per slot, which the training loop reads as N_GROUP

    groups of GROUP consecutive channels.

    """

    studies = sorted(slot_map)

    sidx = {s: i for i, s in enumerate(studies)}

    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)

    mask = np.zeros((len(studies), N_SLOT), np.float32)

    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")



    jobs = [(st, k, plane, slot_map[st][name])

            for st in studies

            for k, (name, plane, _, _) in enumerate(SLOTS)

            if name in slot_map[st]]



    # Ordering first, and as its own pass. It reads one header per slice of every chosen

    # series - far more file opens than the decode that follows - and on a network mount

    # that is latency, not work, so it gets its own wider pool.

    t_ord = time.time()

    n_slice_total = sum(len(j[3]["files"]) for j in jobs)

    log(f"{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)")

    ok = done = 0

    CHUNK_O = 1024

    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:

        for c0 in range(0, len(jobs), CHUNK_O):

            block = jobs[c0:c0 + CHUNK_O]

            for (_, _, _, rec), (files, good) in zip(

                    block, pool.map(lambda j: order_slices(j[3]), block)):

                rec["ordered"] = files

                ok += int(good)

                done += 1

            # The ceiling is whichever comes first: the pass's own budget, or the share

            # of what is left of the run that it may take. The second is what makes the

            # first safe to set generously - a mount slow enough to matter cannot spend

            # the training time, because the budget shrinks as the run does.

            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))

            if time.time() - t_ord > budget:

                log(f"{tag}: ordering budget spent at {done}/{len(jobs)}; "

                    f"the rest keep file order")

                break

    log(f"{tag}: ordered {ok}/{len(jobs)} by geometry "

        f"({len(jobs) - ok} kept arbitrary) in {time.time() - t_ord:.0f}s")



    log(f"{tag}: decoding {len(jobs)} slot-series")

    n_failed_before = len(DECODE_FAILED)



    CHUNK = 512

    done = 0

    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:

        for c0 in range(0, len(jobs), CHUNK):

            block = jobs[c0:c0 + CHUNK]

            for (st, k, plane, _), img in zip(

                    block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):

                done += 1

                if img is None:

                    continue

                cache[sidx[st], k] = normalise_laterality(img, plane,

                                                          lat_map.get(st)).numpy()

                mask[sidx[st], k] = 1.0

            if done % 4096 < CHUNK:

                log(f"  {tag} {done}/{len(jobs)}")

            if time.time() - T0 > TIME_BUDGET:

                log(f"  {tag}: time budget reached during decode")

                break

    n_failed = len(DECODE_FAILED) - n_failed_before

    log(f"{tag}: {int(mask.sum())}/{len(jobs)} slots filled"

        + (f"; {n_failed} series had a slice that would not decode" if n_failed else ""))

    gc.collect()

    return studies, cache, mask


## Step 3 — vision: adaptive encoder (HF | KerasHub | stub) + Gemma report cache

`load_vision()` walks every attached candidate (config.json dirs, shortest
path first). MedGELIP-like layouts pass if they have vision weights; the
branch survives only if a zero-image forward returns the expected width.
No encoder attached? The run still trains: a small learned stub features the
raw pixels (worse, but valid and testable on a laptop).

The Gemma cell embeds every report once per split (train/test only; fold
training reuses the same cache). A GPU-less kernel skips Gemma and the v1
extractor text features remain the report branch.

In [ ]:
"""vision cell for v2 (adaptive MedSigLIP encoder, one encode() interface)."""

# ---- vision: adaptive encoder (HF SigLIP | KerasHub MedSigLIP | stub) ----
# Fast-path contract: when a frozen encoder is present, every slot image is
# encoded ONCE into a (studies, N_SLOT, D) float32 feature cache and the
# image caches are freed; the head then trains on precomputed features,
# which is what makes the GPU part finish in minutes instead of hours.

import os
os.environ.setdefault("KERAS_BACKEND", "torch")   # before any keras import

VISION_MICROBATCH = 8

_INPUT_SCAN = None


def kaggle_input_dirs():
    """One memoised walk of the input root; the scan is otherwise the
    single most wasteful thing on Kaggle (rglob is revisited everywhere)."""
    global _INPUT_SCAN
    if _INPUT_SCAN is None:
        base = Path("/kaggle/input") if Path("/kaggle/input").is_dir() \
            else Path(".")
        _INPUT_SCAN = {str(r) for r in base.rglob("*") if r.is_dir()}
    return _INPUT_SCAN


class VisionEncoder(nn.Module):
    """interface: encode(x [B,C,H,W] in [0,1]) -> (B,D); feature_dim: int"""

    def __init__(self, feature_dim):
        super().__init__()
        self.feature_dim = feature_dim
        self.frozen = True              # frozen encoders unlock the feat cache
        self.grad = False               # True = finetune mode (encoder in train)

    def set_grad(self, trainable_blocks=0):
        """Unfreeze the last trainable_blocks transformer blocks + the
        post-encoder projection, leave the rest frozen. Returns the number
        of trainable parameters (>0 means fine-tuning is live)."""
        self.grad = trainable_blocks > 0
        if self.grad:
            self.frozen = False
        n = 0
        for p in self.parameters():
            if p.requires_grad:
                n += p.numel()
        return n

    def encode(self, x):
        raise NotImplementedError


class HFEncoder(VisionEncoder):
    def __init__(self, model, feature_dim, device):
        super().__init__(feature_dim)
        self.model = model.to(device)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    def _blocks(self):
        """-> (vision stream, list of transformer blocks) or (None, [])."""
        m = self.model
        if hasattr(m, "vision_model"):
            vs = m.vision_model
        elif hasattr(m, "vision_encoder"):
            vs = m.vision_encoder
        else:
            vs = m
        if hasattr(vs, "encoder") and hasattr(vs.encoder, "layers"):
            return vs, list(vs.encoder.layers)
        return vs, []

    def set_grad(self, trainable_blocks=0):
        for p in self.model.parameters():
            p.requires_grad_(False)
        vs, blocks = self._blocks()
        n = 0
        if trainable_blocks > 0 and len(blocks):
            for b in blocks[-trainable_blocks:]:
                for p in b.parameters():
                    p.requires_grad_(True)
                    n += p.numel()
        # the pooled projection feeds the head; unfreeze with the blocks
        if hasattr(vs, "pooler") and n:
            for p in vs.pooler.parameters():
                p.requires_grad_(True)
                n += p.numel()
        elif hasattr(vs, "post_layernorm") and n:
            for p in vs.post_layernorm.parameters():
                p.requires_grad_(True)
                n += p.numel()
        self.grad = n > 0
        self.frozen = not self.grad
        return n

    def encode(self, x):
        if self.grad:
            self.model.train()
            out = self.model(pixel_values=x).last_hidden_state
        else:
            with torch.no_grad():
                self.model.eval()
                out = self.model(pixel_values=x).last_hidden_state
        return out.mean(1)


class KerasEncoder(VisionEncoder):
    """keras_hub preset behind the same encode() contract (torch backend)."""

    def __init__(self, backbone, converter, feature_dim, device):
        super().__init__(feature_dim)
        self.backbone = backbone
        self.converter = converter
        for layer in getattr(backbone, "_layers", []):
            try:
                layer.trainable = False
            except Exception:
                pass
        try:
            self.backbone.to(device)
        except Exception:
            pass
        self.backbone.eval()
        self.device = device

    def set_grad(self, trainable_blocks=0):
        """KerasEncoder fine-tuning is NOT supported: the keras-hub eager
        graph can't be half-unfrozen without a verified recompile, and
        unfreezing all layers at 448px would blow T4 memory. The HF encoder
        path carries fine-tuning; Keras stays on the frozen feature cache.
        Returns 0 so the caller falls back to the feats path."""
        self.grad = False
        self.frozen = True
        return 0

    def _vision(self, x):
        if hasattr(self.backbone, "vision_encoder"):
            return self.backbone.vision_encoder
        for cand in ("vision", "vision_model", "image_encoder", "visual"):
            if hasattr(self.backbone, cand):
                return getattr(self.backbone, cand)
        raise AttributeError("no vision stream on preset")

    def _pick(self, y):
        if isinstance(y, dict):
            for k in ("sequence_embedding", "pooled_embedding"):
                if k in y and y[k] is not None:
                    return y[k]
            return next(iter(y.values()))
        if hasattr(y, "sequence_embedding"):
            return y.sequence_embedding
        if hasattr(y, "pooled_embedding"):
            return y.pooled_embedding
        return y

    def encode(self, x):
        x = (x * 255.0).byte().permute(0, 2, 3, 1)      # B,H,W,C uint8
        with torch.no_grad():
            pre = self.converter(x)
            try:
                y = self._vision(pre) if not isinstance(pre, dict) else \
                    self._vision({"images": pre["images"]})
            except TypeError:
                y = self._vision({"images": pre if not isinstance(pre, dict)
                                  else pre["images"]})
            if isinstance(y, dict) or hasattr(y, "sequence_embedding"):
                y = self._pick(y)
            t = torch.as_tensor(y, device=self.device)
            if t.dim() == 3:
                t = t.mean(1)
        return t.float()


class StubEncoder(VisionEncoder):
    def __init__(self, dim, device):
        super().__init__(dim)
        self.frozen = False          # trainable: keep the pixel pipeline
        self.model = nn.Sequential(nn.AdaptiveAvgPool2d((32, 32)), nn.Flatten(),
                                   nn.Linear(3 * 32 * 32, dim)).to(device)
        self.device = device

    def set_grad(self, trainable_blocks=0):
        n = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        self.grad = n > 0
        self.frozen = not self.grad
        return n

    def encode(self, x):
        return self.model(x)


class FeatBackbone(VisionEncoder):
    """Header-only marker for the feature-cache path: the heavy encoder is
    burned into feat_cache_*.npz; models just need the width, not the
    weights (which would otherwise be duplicated per fold)."""

    def __init__(self, feature_dim, device):
        super().__init__(feature_dim)
        self.device = device

    def set_grad(self, trainable_blocks=0):
        return 0                       # features already cached by the caller

    def encode(self, x):
        raise RuntimeError("FeatBackbone is never meant to encode; "
                           "feed precomputed feats instead")

    def to(self, *a, **k):
        return self


def _vision_dirs():
    """config.json-bearing dirs under the input root."""
    out = []
    for s in sorted(kaggle_input_dirs(), key=len):
        if Path(s) / "config.json" is not None and \
                (Path(s) / "config.json").is_file():
            out.append(Path(s))
    return out


def _smoke(enc, device):
    """A zero-image forward at load time; a failed branch must not survive."""
    try:
        dev = next(enc.parameters()).device
    except StopIteration:
        dev = device
    with torch.no_grad():
        y = enc.encode(torch.zeros(2, 3, IMG, IMG, device=dev))
    if y.shape[0] != 2 or y.shape[1] != enc.feature_dim:
        raise ValueError("smoke shape mismatch")
    return enc


def _kerashub_one(d, device):
    """One from_preset attempt; raises on any failure."""
    import keras
    import keras_hub
    keras.config.set_backend("torch")
    backbone = keras_hub.models.Backbone.from_preset(str(d))
    converter = keras_hub.layers.SigLIPImageConverter.from_preset(str(d))
    dim = int(getattr(backbone, "hidden_dim", None) or 1152)
    return KerasEncoder(backbone, converter, dim, device)


def _kerashub_load(d, device):
    """KerasHub preset -> KerasEncoder, with one pip-upgrade retry.

    The Kaggle images keep shipping keras a bit older than the preset's
    serialization; a single upgrade+reimport, gated behind the network
    probe, fixes the DTypePolicy deserialization error in almost all cases.
    """
    import socket, subprocess, sys

    def net_up():
        try:
            socket.create_connection(("pypi.org", 443), timeout=2).close()
            return True
        except Exception:
            return False

    try:
        return _kerashub_one(d, device)
    except Exception as e:
        log(f"vision: KerasHub load failed: {str(e)[:160]}")
    if not net_up():
        raise RuntimeError("keras preset mismatch and no internet for pip")
    try:
        log("vision: upgrading keras/keras-hub (pypi reachable)")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--upgrade", "keras", "keras-hub"],
                       capture_output=True, timeout=600)
        for m in list(sys.modules):
            if m == "keras" or m.startswith("keras.") or \
               m == "keras_hub" or m.startswith("keras_hub.") or \
               m == "keras_hub_io" or m.startswith("keras_hub_io."):
                del sys.modules[m]
        try:
            import keras
            import keras_hub
            log(f"keras {keras.__version__} keras_hub {keras_hub.__version__}")
        except Exception as e:
            log(f"vision: keras reimport failed: {e!r}")
        return _kerashub_one(d, device)
    except Exception as e2:
        log(f"vision: KerasHub retry failed: {str(e2)[:160]}")
        raise


def load_vision(device):
    """-> VisionEncoder, never None (last resort = trainable stub).

    HF SiglipVisionModel dirs are tried first (they unlock the same frozen
    feature-cache fast path), then KerasHub presets (with an upgrade retry),
    then the stub - which only trains on pixels and is noticeably slower.
    """
    for d in _vision_dirs():
        try:
            cfg = json.load(open(d / "config.json"))
        except Exception:
            continue
        if cfg.get("model_type") != "siglip":
            continue
        try:
            from transformers import SiglipVisionModel
            m = SiglipVisionModel.from_pretrained(str(d))
            vc = cfg.get("vision_config") or cfg
            dim = vc.get("hidden_size", 768)
            return _smoke(HFEncoder(m, dim, device), device)
        except Exception as e:
            log(f"vision: HF load failed: {e!r}")

    for d in _vision_dirs():
        if not (d / "model.weights.h5").is_file():
            continue
        try:
            return _smoke(_kerashub_load(d, device), device)
        except Exception:
            log(f"vision: KerasHub preset unusable: {str(d)[-60:]}")

    log("vision: no encoder attached - using stub (random init, trainable)")
    return _smoke(StubEncoder(256, device), device)


def build_feature_cache(enc, cache, uids, tag, dev):
    """Encode every cached slot once -> (len(uids), N_SLOT, D) float32.

    The cache lives on disk (feat_cache_<tag>.npz) and is loaded back on
    reruns; on the fast path the caller then drops the image caches and
    trains on the features. Returns None when the encoder is trainable
    (stub) - the pixel pipeline stays in charge in that case.
    """
    if enc is None or getattr(enc, "frozen", False) is False:
        log(f"feats[{tag}]: encoder is trainable - keeping image pipeline")
        return None
    fp = Path(f"feat_cache_{tag}.npz")
    try:
        z = np.load(fp)
        if list(z["uid"]) == list(uids):
            log(f"feats[{tag}]: hit {fp.name} {z['f'].shape}")
            return z["f"].astype(np.float32)
    except Exception:
        pass
    M = len(uids)
    D = enc.feature_dim
    rows = cache[:, :, :GROUP]        # (M, S, 3, H, W) uint8
    f = np.zeros((M, rows.shape[1], D), np.float32)
    step = 16
    enc_dev = torch.device("cpu")
    for p in enc.parameters():
        enc_dev = p.device
        break
    use_amp = enc_dev.type == "cuda"
    for i0 in range(0, M, step):
        batch = rows[i0:i0 + step]
        b, s = batch.shape[:2]
        x = torch.from_numpy(batch).reshape(b * s, *batch.shape[2:]) \
            .float().div_(255.0)
        out = []
        for j in range(0, x.shape[0], VISION_MICROBATCH):
            chunk = x[j:j + VISION_MICROBATCH].to(enc_dev)
            with torch.autocast("cuda", dtype=torch.bfloat16) if use_amp \
                    else torch.no_grad():
                out.append(enc.encode(chunk))
        f[i0:i0 + step] = torch.cat(out, 0).reshape(b, s, D).cpu().numpy()
        if (i0 // step) % 8 == 0:
            log(f"feats[{tag}]: {i0 + len(batch)}/{M} slots encoded")
    np.savez(fp, uid=np.array(uids, dtype=object), f=f.astype(np.float16))
    log(f"feats[{tag}]: cached {f.shape} -> {fp.name}")
    return f

In [ ]:
"""text cell for v2: adaptive Gemma-4 loader + one-shot embedding cache."""

# ---- text: Gemma-4 report embeddings (offline, once per study) ----------

TEXT_MAXLEN = 96


def find_model_dir(keys):
    base = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path(".")
    best = None
    for d in base.rglob("*"):
        if not d.is_dir():
            continue
        if not (d / "config.json").is_file():
            continue
        name = str(d).lower()
        if not any(k in name for k in keys):
            continue
        has_w = (d / "model.safetensors").is_file() or \
                (d / "model.weights.h5").is_file()
        if has_w:
            best = d if best is None else min(best, d, key=lambda p: len(str(p)))
    return best


GEMMA_DIR = find_model_dir(("gemma",))
log(f"Gemma: {str(GEMMA_DIR)}" if GEMMA_DIR else "Gemma NOT attached")


def load_text_model():
    """-> (tokenizer, model) or (None, None). CUDA-only; never raises."""
    if GEMMA_DIR is None:
        return None, None
    try:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(str(GEMMA_DIR))
    except Exception as e:
        log(f"text: tokenizer failed: {e!r}")
        return None, None
    cfg = {}
    try:
        cfg = json.load(open(GEMMA_DIR / "config.json"))
    except Exception:
        pass
    arch = (cfg.get("architectures") or [""])[0]
    try:
        import torch
        try:
            import bitsandbytes  # noqa: F401
            has_bnb = True
        except Exception:
            has_bnb = False
        if has_bnb:
            try:
                if "ConditionalGeneration" in arch:
                    from transformers import AutoModelForMultimodalLM as Klass
                else:
                    from transformers import AutoModelForCausalLM as Klass
                m = Klass.from_pretrained(str(GEMMA_DIR), load_in_4bit=True,
                                          device_map="auto",
                                          torch_dtype=torch.bfloat16)
                log(f"  Gemma loaded 4-bit ({arch or 'auto'})")
                return tokenizer, m
            except Exception as e:
                log(f"  4-bit failed: {str(e)[:160]}")
        try:
            if "ConditionalGeneration" in arch:
                from transformers import AutoModelForMultimodalLM as Klass
            else:
                from transformers import AutoModelForCausalLM as Klass
            m = Klass.from_pretrained(str(GEMMA_DIR), device_map="auto",
                                      torch_dtype=torch.bfloat16)
            log(f"  Gemma loaded bf16 ({arch or 'auto'})")
            return tokenizer, m
        except Exception as e:
            log(f"  bf16 failed: {str(e)[:160]}")
    except Exception as e:
        log(f"  torch/transformers unavailable: {e!r}")
    log("  Gemma unavailable - extractor T0 features carry the text branch")
    return None, None


def text_embed_batch(model, tokenizer, texts, dev):
    import torch.nn.functional as F
    enc = tokenizer(texts, padding="max_length", truncation=True,
                    max_length=TEXT_MAXLEN, return_tensors="pt")
    ids, am = enc["input_ids"].to(dev), enc["attention_mask"].to(dev)
    with torch.no_grad():
        if hasattr(model, "language_model"):
            out = model.language_model(input_ids=ids, attention_mask=am,
                                       output_hidden_states=True)
        else:
            out = model(input_ids=ids, attention_mask=am,
                        output_hidden_states=True)
        h = out.hidden_states[-1]
        mask = am.unsqueeze(-1).to(h.dtype)
        pooled = (h * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        pooled = F.normalize(pooled, p=2, dim=-1)
    return pooled.float().cpu().numpy()


def build_text_cache(uids, reports, dev, tag):
    """(len(1), D) float32, keyed by StudyInstanceUID, cached as npz.

    Cache is keyed by tag (train/test) so the two uids lists never collide.
    dev must be 'cuda' when the cache is missing: the model is ~8B params and
    device_map auto flattens onto CPU when the GPU is absent; on CPU the
    extractor text features from v1 remain the report branch instead.
    """
    fp = Path(f"text_cache_{tag}.npz")
    try:
        z = np.load(fp)
        if list(z["uid"]) == list(uids):
            log(f"text[{tag}]: cache hit {fp.name} ({z['emb'].shape})")
            return z["emb"].astype(np.float32)
    except Exception:
        pass
    if dev.type != "cuda":
        log(f"text[{tag}]: GPU required for Gemma - skipped (T0 carries text)")
        return None
    tokenizer, model = load_text_model()
    if tokenizer is None:
        return None
    log(f"text[{tag}]: embedding {len(uids)} studies")
    D = None
    embs = np.zeros((len(uids), 3840), np.float32)
    B = 8
    for b in range(0, len(uids), B):
        texts = [str(reports.get(u, "")) for u in uids[b:b + B]]
        e = text_embed_batch(model, tokenizer, texts, dev)
        if D is None:
            D = e.shape[1]
            embs = np.zeros((len(uids), D), np.float32)
        embs[b:b + B, :D] = e
    np.savez(fp, uid=np.array(uids, dtype=object), emb=embs.astype(np.float16))
    log(f"text[{tag}]: cached {embs.shape} -> {fp}")
    return embs

In [ ]:
"""model cell for v2: V2Model = vision + text + fusion -> SlotHead (v1 verbatim)."""

# ---- model: SlotHead (v1) + multimodal fusion ---------------------------

class SlotHead(nn.Module):
    """Per-diagnosis attention over the slot embeddings of one study."""

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class V2Model(nn.Module):
    """vision.encode -> (B,S,Dv); report text broadcast per slot;
    concat -> fusion MLP -> SlotHead (v1, untouched)."""

    def __init__(self, vision, text_dim=0, feature_dim=None, fusion_dim=256, p=0.1):
        super().__init__()
        self.vision = vision
        self.text_dim = int(text_dim or 0)
        dv = feature_dim or getattr(vision, "feature_dim", 256)
        self.vision_dim = dv
        self.use_text = self.text_dim > 0
        self.fusion = nn.Sequential(
            nn.Linear(dv + (self.text_dim if self.use_text else 0), fusion_dim),
            nn.LayerNorm(fusion_dim), nn.GELU(), nn.Dropout(p))
        self.head = SlotHead(fusion_dim, N_SLOT, len(TARGETS))

    def encode_slots(self, imgs):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        feats = []
        for i in range(0, x.shape[0], VISION_MICROBATCH):
            chunk = x[i:i + VISION_MICROBATCH]
            if isinstance(self.vision, StubEncoder):
                feats.append(self.vision.encode(chunk))
            elif self.vision.grad:           # fine-tune mode: backprop through
                feats.append(self.vision.encode(chunk))
            else:
                with torch.no_grad():
                    feats.append(self.vision.encode(chunk))
        return torch.cat(feats, 0).reshape(B, S, -1)

    def forward(self, imgs, mask, txt=None, feats=None):
        if feats is None:
            v = self.encode_slots(imgs)
        else:
            v = feats
        if self.use_text:
            if txt is None:
                txt = torch.zeros(v.shape[0], self.text_dim, device=v.device)
            v = torch.cat([v, txt.unsqueeze(1).expand(-1, v.shape[1], -1)], dim=-1)
        return self.head(self.fusion(v), mask)


class Ema:
    """Shadow EMA of trainable params (no deepcopy - safe with Keras encoder).

    skip_prefixes: when fine-tuning the vision stack, EMA tracks only the
    head/fusion (the encoder already averages through its own slow LR);
    shadowing 100M+ encoder params per step would dominate the clock.
    """

    def __init__(self, model, decay, skip_prefixes=()):
        self.decay = decay
        self.step = 0
        self.shadow = {
            n: p.detach().cpu().clone()
            for n, p in model.named_parameters() if p.requires_grad
            and not n.startswith(skip_prefixes)}

    @torch.no_grad()
    def update(self, model):
        if self.decay <= 0:
            return
        self.step += 1
        decay = min(self.decay, (1.0 + self.step) / (10.0 + self.step))
        for name, p in model.named_parameters():
            if p.requires_grad and name in self.shadow:
                s = self.shadow[name].to(p.device)
                s.mul_(decay).add_(p.detach(), alpha=1.0 - decay)

    @torch.no_grad()
    def target(self, model):
        for name, p in model.named_parameters():
            if p.requires_grad and name in self.shadow:
                p.copy_(self.shadow[name].to(p.device))
        return model


def build_v2_model(vision, text_dim=0, feature_dim=None, fusion_dim=256):
    if isinstance(vision, StubEncoder):
        from copy import deepcopy
        vision = deepcopy(vision)     # fresh stub per model (v1 behaviour)
    return V2Model(vision, text_dim=text_dim, feature_dim=feature_dim,
                   fusion_dim=fusion_dim)


def build_eval_model(vision, text_dim, dev, src=None, feat_dim=None):
    """Fresh V2Model; EMA/vision weights copied in (no shared state)."""
    m = build_v2_model(vision, text_dim,
                       feature_dim=feat_dim or getattr(vision, "feature_dim",
                                                       None)).to(dev)
    if src is not None:
        m.load_state_dict(src.state_dict(), strict=False)
    return m

## Step 4 — slot-attention head (v1 verbatim), shadow EMA, CV, submission

The head is unchanged from v1: per-diagnosis attention over the six slot
views. `V2Model` adds the fusion (vision + optional text) before the head;
`Ema` tracks shadow averages of the trainable weights and `build_eval_model`
copies them into a fresh eval model — never mutating the training instance.
Augmentation stays pixel-space and runs BEFORE the encoder, exactly like v1.
CV folds, rank loss and the multi-fold rank-averaged submission are v1.

In [ ]:
"""augment cell for v2: v1 pixel-space aug + rank loss kept verbatim."""

def augment(imgs):

    """Small affine + intensity jitter over the whole bag. No flips."""

    B, S, C, H, W = imgs.shape

    x = imgs.float()

    ang = math.radians((np.random.rand() - 0.5) * 2 * AUG_ROT_DEG)

    sc = 1.0 + (np.random.rand() - 0.5) * 2 * AUG_SCALE

    tx = (np.random.rand() - 0.5) * 2 * AUG_SHIFT

    ty = (np.random.rand() - 0.5) * 2 * AUG_SHIFT

    cos, sin = math.cos(ang) / sc, math.sin(ang) / sc

    theta = torch.tensor([[cos, -sin, tx], [sin, cos, ty]], dtype=torch.float32)

    theta = theta.unsqueeze(0).repeat(B * S, 1, 1)

    flat = x.reshape(B * S, C, H, W)

    grid = F.affine_grid(theta, flat.shape, align_corners=False)

    x = F.grid_sample(flat, grid, mode="bilinear", padding_mode="zeros",

                      align_corners=False).reshape(B, S, C, H, W)

    scale = 1.0 + (np.random.rand() - 0.5) * 2 * AUG_INTENSITY

    x = (x * scale).clamp(0, 255)

    if USE_VFLIP and np.random.rand() < 0.5:

        x = torch.flip(x, dims=[-2])

    return x


def rank_loss(logits, y, w):

    parts = []

    usable = w > 0

    for j in range(logits.shape[1]):

        pos = logits[(y[:, j] > RANK_POS) & usable[:, j], j]

        neg = logits[(y[:, j] < RANK_NEG) & usable[:, j], j]

        if len(pos) and len(neg):

            parts.append(F.softplus(-(pos[:, None] - neg[None, :])).mean())

    return torch.stack(parts).mean() if parts else logits.new_tensor(0.0)

In [ ]:
"""predict cell for v2: v1 predict (now text-aware) + submission writers."""

def predict(model, cache, mask, idx, dev, txt=None, feats=False):
    """Logits (n,12), averaging the cached groups.

    feats=True: `cache` carries precomputed (n, N_SLOT, Dv) features
    (feature-cache fast path); otherwise it is the uint8 image cache.
    """

    model.eval()
    out = []
    with torch.no_grad():
        for b in range(0, len(idx), EVAL_BATCH):
            sel = idx[b:b + EVAL_BATCH]
            m = torch.from_numpy(mask[sel]).to(dev)
            t = (torch.from_numpy(txt[sel]).float().to(dev)
                 if txt is not None else None)
            if feats:
                v = torch.from_numpy(cache[sel]).float().to(dev)
                z = model(None, m, t, feats=v)
            else:
                rows = torch.from_numpy(cache[sel]).to(dev)
                imgs = rows[:, :, :GROUP]
                z = model(imgs, m, t)
            out.append(z.float().cpu().numpy())
    return np.concatenate(out)


def macro_auc(y, p):
    from sklearn.metrics import roc_auc_score

    s = []
    for j in range(p.shape[1]):
        if len(np.unique(y[:, j])) == 2 and np.sum(y[:, j] == 1) > 0:
            try:
                s.append(roc_auc_score(y[:, j], p[:, j]))
            except ValueError:
                pass
    return float(np.mean(s)) if s else float("nan")


def write_benchmark_submission():
    t = pd.read_csv(ROOT / "test.csv")
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv("submission.csv", index=False)


def write_submission(rank_sum, n_models, st_te, test_df):
    P = rank_sum / max(n_models, 1)
    sub = pd.DataFrame(P, columns=TARGETS)
    sub.insert(0, "StudyInstanceUID", st_te)
    sub = test_df[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv("submission.csv", index=False)
    return sub

In [ ]:
"""main cell for v2 (ceiling path): clean supervision, CV on confident
labels, and optional vision fine-tuning during the fold loop."""

# ---- orchestrator --------------------------------------------------------

REQUIRE_GPU = True

# --- ceiling levers -------------------------------------------------------
TGT_CONF_FLOOR = 0.45   # drop lexicon pseudo-labels below this conf (noise)
CLEAN_CONF = 0.50       # validation AUC only on targets at/above conf
FINE_TUNE = True        # unfreeze the last N vision blocks during folds
FINE_TUNE_BLOCKS = 6
FT_LR = 3e-5            # encoder-block LR (head keeps LR_HEAD)
FT_EPOCHS = 2           # fine-tune epochs per fold (pixel path)
EPOCHS_FAST = 4         # frozen-feature-path epochs per fold


def tune_threads():
    """Scale the v1 thread pools to this instance's vCPUs. The DICOM->cache
    phase is the run's floor (thread-bound), so oversubscription of the
    actual core count is the only lever that moves it."""
    try:
        n = os.cpu_count() or 8
        globals()["HDR_THREADS"] = max(HDR_THREADS, min(n, 64))
        globals()["PIX_THREADS"] = max(PIX_THREADS, min(n, 48))
        globals()["ORDER_THREADS"] = max(ORDER_THREADS, min(n * 2, 96))
        os.environ["OMP_NUM_THREADS"] = str(min(n, 16))
        os.environ["OPENBLAS_NUM_THREADS"] = str(min(n, 16))
        log(f"threads: hdr {HDR_THREADS} pix {PIX_THREADS} "
            f"order {ORDER_THREADS} (vCPUs {n})")
    except Exception as e:
        log(f"thread tuning failed: {e!r}")


def build_clean_targets(st_tr, gold, lab, floor=TGT_CONF_FLOOR):
    """Y/W as in v1, plus the per-target confidence `C` (1.0 for gold).

    Lexicon pseudo-labels whose extractor confidence is below the floor get
    weight 0 - they are noise, not supervision; gold labels are left at
    GOLD_WEIGHT. keep = rows carrying at least one supervised target.
    """
    Y = np.zeros((len(st_tr), len(TARGETS)), np.float32)
    W = np.zeros_like(Y)
    C = np.zeros_like(Y)
    for i, st in enumerate(st_tr):
        if st in gold.index:
            Y[i], W[i] = gold.loc[st].values, GOLD_WEIGHT
            C[i] = 1.0
        elif st in lab.index:
            r = lab.loc[st]
            Y[i] = r[TARGETS].values
            C[i] = r[[t + "__conf" for t in TARGETS]].values
            W[i] = 0.25 + 0.75 * C[i]
            W[i][C[i] < floor] = 0.0
    return Y, W, W.sum(1) > 0, C


def clean_auc(y, p, conf, floor):
    """macro-AUC over the confident subset: only (sample,target) pairs
    whose extractor confidence >= floor and both classes are present.
    Targets are binarised like v1 (pseudo-labels are graded scores)."""
    from sklearn.metrics import roc_auc_score
    yb = (y > 0.5).astype(int)
    s, n = [], 0
    for j in range(p.shape[1]):
        m = conf[:, j] >= floor
        if m.sum() < 16 or len(np.unique(yb[m, j])) < 2:
            continue
        try:
            s.append(roc_auc_score(yb[m, j], p[m, j]))
            n += 1
        except ValueError:
            continue
    return (float(np.mean(s)) if s else float("nan")), n


def main():
    tune_threads()                             # before the DICOM phase
    write_benchmark_submission()             # scoreable from second 1

    def pick_device():
        # CUDA if a real kernel is runnable, else CPU.
        # Recent torch builds dropped Pascal (P100, sm_60) kernels, so
        # cuda.is_available() can be False even on GPU machines; and on
        # older builds the first forward can die with "no kernel image is
        # available". A probe op + explicit logging catches both.
        import torch as _t
        if not _t.cuda.is_available():
            log(f"pick_device: no CUDA build "
                f"(torch {_t.__version__}) -> cpu")
            return "cpu"
        try:
            _t.ones(2, device="cuda").float().sum().item()
            _t.cuda.synchronize()
        except Exception as e:
            log(f"pick_device: probe failed ({type(e).__name__}: "
                f"{str(e)[:80]}) -> cpu")
            return "cpu"
        log(f"pick_device: cuda ({_t.cuda.get_device_name(0)})")
        return "cuda"

    dev = torch.device(pick_device())
    if REQUIRE_GPU and dev.type != "cuda":
        log("GPU required but not runnable; keeping the 0.5 benchmark "
            "submission and stopping now (no DICOM burn, no CPU burn). "
            "Pick a T4 GPU on Kaggle, or an older container image "
            "(torch with Pascal kernels) on a P100.")
        return
    log(f"compute device: {dev}"
        f"{' (' + torch.cuda.get_device_name(0) + ')' if dev.type == 'cuda' else ''}")

    test_df = pd.read_csv(ROOT / "test.csv")
    test_series = pd.read_csv(ROOT / "test_series.csv")
    train_df = pd.read_csv(ROOT / "train.csv")
    train_series = pd.read_csv(ROOT / "train_series.csv")
    log(f"train {train_df.shape} test {test_df.shape}")

    both = pd.concat([train_series, test_series])
    plane_map = dict(zip(both["SeriesInstanceUID"], both["Anatomical_Plane"]))

    htr = annotate(walk("train_series"))
    hte = annotate(walk("test_series"))
    lat_tr, _ = laterality_maps(htr)
    lat_te, _ = laterality_maps(hte)
    slots_tr = pick_slots(htr, plane_map)
    slots_te = pick_slots(hte, plane_map)

    st_tr, Ctr, Mtr = build_cache(slots_tr, plane_map, lat_tr, "train")
    st_te, Cte, Mte = build_cache(slots_te, plane_map, lat_te, "test")

    gold = train_df.set_index("StudyInstanceUID")[TARGETS]
    gold = gold[gold.notna().all(axis=1)]
    lab = read_labels(train_df)
    Y, W, keep, Conf = build_clean_targets(st_tr, gold, lab, TGT_CONF_FLOOR)
    log(f"supervised {int(keep.sum())} of {len(st_tr)} studies "
        f"(conf floor {TGT_CONF_FLOOR})")

    rep = train_df.set_index("StudyInstanceUID")["Report"].fillna("")
    grp = np.array([int(hashlib.md5(str(rep.get(s, s)).encode()).hexdigest()[:8], 16)
                    % N_FOLDS for s in st_tr])
    gpos = {s: i for i, s in enumerate(st_tr)}
    gi_all = np.array([gpos[s] for s in gold.index if s in gpos])

    vision = load_vision(dev)
    use_ft = (FINE_TUNE and isinstance(vision, HFEncoder)
              and dev.type == "cuda")
    if use_ft:
        nft = vision.set_grad(FINE_TUNE_BLOCKS)
        log(f"fine-tune ON: last {FINE_TUNE_BLOCKS} blocks trainable "
            f"({nft:,} params), {FT_EPOCHS} epochs/fold")
    else:
        log(f"fine-tune OFF: frozen-encoder feats path "
            f"(encoder {type(vision).__name__})")

    te_tr = build_text_cache(st_tr, rep.to_dict(), dev, "train")
    try:
        rep_te = test_df.set_index("StudyInstanceUID")["Report"].fillna("")
        te_te = build_text_cache(st_te, rep_te.to_dict(), dev, "test")
    except Exception:
        te_te = None
    text_dim = (te_tr.shape[1] if te_tr is not None else 0)
    if text_dim:
        log(f"text: {text_dim}-d report embeddings (train {te_tr is not None}, "
            f"test {te_te is not None})")
    else:
        log("text: no Gemma embeddings - extractor T0 is the text branch")

    feats_tr = feats_te = None
    if use_ft:
        # fine-tuning needs the pixel path: no feature cache, keep the
        # image caches alive for the fold loop
        _EPOCHS = FT_EPOCHS
    else:
        feats_tr = build_feature_cache(vision, Ctr, st_tr, "train", dev)
        feats_te = build_feature_cache(vision, Cte, st_te, "test", dev) \
            if feats_tr is not None else None
        if feats_tr is not None and feats_te is not None:
            del Ctr, Cte
            gc.collect()
            log("feature cache active - image caches freed")
            from copy import deepcopy
            vision = deepcopy(FeatBackbone(vision.feature_dim, dev))
            _EPOCHS = EPOCHS_FAST
        else:
            _EPOCHS = EPOCHS
    log(f"encoder {type(vision).__name__} dim={vision.feature_dim} "
        f"epochs/fold={_EPOCHS}")

    rank_sum = np.zeros((len(st_te), len(TARGETS)), np.float64)
    n_models, sub = 0, None

    for fold in range(N_FOLDS):
        va = np.array([i for i in range(len(st_tr))
                       if keep[i] and grp[i] == fold])
        tr = np.array([i for i in range(len(st_tr))
                       if keep[i] and grp[i] != fold])
        if len(va) == 0 or len(tr) < 2:
            log(f"fold {fold}: too small, skipped"); continue
        gi_va = np.array([i for i in gi_all if grp[i] == fold])
        log(f"=== fold {fold}: train {len(tr)} holdout {len(va)}"
            f" (gold held out {len(gi_va)}) ===")
        t0 = time.time()
        model = build_v2_model(vision, text_dim).to(dev)
        # two LR groups in fine-tune mode: encoder (slow) vs head (fast)
        body = [p for n_, p in model.named_parameters()
                if p.requires_grad and n_.startswith("vision.")]
        head = [p for p in model.parameters() if p.requires_grad
                and all(p is not q for q in body)]
        groups = [{"params": head, "lr": LR_HEAD}]
        if use_ft and body:
            groups = [{"params": body, "lr": FT_LR}] + groups
        opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
        ema = Ema(model, EMA_DECAY,
                  skip_prefixes=("vision.",) if use_ft else ())
        steps = max(_EPOCHS * max(len(tr) // BATCH_STUDIES, 1), 1)
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=[g["lr"] for g in groups],
            total_steps=steps, pct_start=0.15)
        yv = (Y[va] > 0.5).astype(int)
        best, best_state = -1.0, None

        for ep in range(_EPOCHS):
            model.train()
            perm = np.random.permutation(tr)
            tot = nst = 0
            for b in range(0, len(perm) - BATCH_STUDIES + 1, BATCH_STUDIES):
                sel = perm[b:b + BATCH_STUDIES]
                m_ = torch.from_numpy(Mtr[sel]).to(dev)
                y = torch.from_numpy(Y[sel]).to(dev)
                w_ = torch.from_numpy(W[sel]).to(dev)
                t_ = (torch.from_numpy(te_tr[sel]).float().to(dev)
                      if te_tr is not None else None)
                if feats_tr is not None:
                    feats = torch.from_numpy(feats_tr[sel]).to(dev)
                    feats = feats + torch.randn_like(feats) * 0.03
                    z = model(None, m_, t_, feats=feats)
                else:
                    rows = torch.from_numpy(Ctr[sel]).to(dev)
                    imgs = augment(rows[:, :, :GROUP])
                    z = model(imgs, m_, t_)
                loss = (F.binary_cross_entropy_with_logits(z, y, reduction="none")
                        * w_).mean()
                if RANK_LOSS_W > 0:
                    loss = loss + RANK_LOSS_W * rank_loss(z.float(), y, w_)
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
                ema.update(model)
                tot += float(loss); nst += 1
            em = build_eval_model(vision, text_dim, dev, src=model)
            ema.target(em)
            if feats_tr is not None:
                pv = predict(em, feats_tr, Mtr, va, dev, te_tr, feats=True)
            else:
                pv = predict(em, Ctr, Mtr, va, dev, te_tr)
            d, ns = clean_auc(yv, pv, Conf, CLEAN_CONF)
            log(f"fold {fold} ep {ep + 1}/{_EPOCHS} "
                f"loss {tot / max(nst, 1):.4f} clean {d:.4f} ({ns} targets)")
            if d > best:
                best = d
                best_state = {k: v.detach().cpu().clone()
                              for k, v in em.state_dict().items()}
            if time.time() - T0 > TIME_BUDGET:
                log("time budget reached"); break

        if best_state is None:
            em = build_eval_model(vision, text_dim, dev, src=model)
            ema.target(em)
            best_state = {k: v.detach().cpu().clone()
                          for k, v in em.state_dict().items()}
        model = build_v2_model(vision, text_dim).to(dev)
        model.load_state_dict(best_state, strict=False)
        if feats_te is not None:
            P = predict(model, feats_te, Mte, np.arange(len(st_te)), dev,
                        te_te, feats=True)
        else:
            P = predict(model, Cte, Mte, np.arange(len(st_te)), dev, te_te)
        rank_sum += pd.DataFrame(P).rank(pct=True).values
        n_models += 1
        sub = write_submission(rank_sum, n_models, st_te, test_df)
        log(f"fold {fold} done in {time.time() - t0:.0f}s")

    if sub is None:
        log("no fold finished; the 0.5 benchmark submission stands")
        return
    log(f"submission.csv {sub.shape}; models {n_models}")
    print(sub.head().to_string())


try:
    main()
except Exception:
    traceback.print_exc()
    log("run failed; the 0.5 benchmark submission is on disk")